# Explore the covers with visual embeddings

Use a pretrained **DINOv2 Vision Transformer** to explore similarities between TikTok covers. Embed the images once, cluster their features, then browse a map and inspect groups. A cluster is an invitation to look more closely; it is not automatically a frame, topic or reliable indication of AI generation.

This first exploration uses **covers only**, one per video post. Photo posts without a cover are outside this selection. The coverage table makes this visible. We keep the full cover when resizing, including text near its edges; text recognition itself belongs in the OCR workflow.

Run **04_build_tables** after preprocessing. This notebook reads the shared tables and covers, and saves results to your home folder. Its interactive viewer is a self-contained HTML file with thumbnails: no PixPlot, extra web service or open port is required.

Background: [DINOv2 documentation](https://huggingface.co/docs/transformers/model_doc/dinov2), [DINOv2 paper](https://arxiv.org/abs/2304.07193), and [Social Media Lab: Visual Exploration](https://social-media-lab.net/image-analysis/exploration.html). GPL-3.0; the code license does not cover corpus data.

In [ ]:
%pip install -q --no-warn-script-location transformers==5.23.0 huggingface-hub==2.4.0 umap-learn==0.6.0 hdbscan==0.8.44 bokeh==3.10.0 pillow==13.0.0 scikit-learn==1.9.1

import importlib, site
site.addsitedir(site.getusersitepackages())
importlib.invalidate_caches()

## 1. Choose the covers

CPU is the default so preprocessing can use the shared GPU. Change `DEVICE` to `cuda` in a GPU workspace only when the GPU is available. On the seminar server, weights are loaded from the administrator-prepared, read-only `/models/exploration` mount. No weights are copied into your home folder. Outside this setup, the first run downloads to your private Hugging Face cache. Covers are processed locally.

The small DINOv2 model is a practical starting point. Model choice changes the meaning of similarity. Revisit it after inspecting examples rather than assuming one embedding model captures every relevant feature.

In [ ]:
import base64, hashlib, html, io, json, os, re
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from IPython.display import display, HTML, FileLink

TABLES = Path(os.environ.get("SEMINAR_OUTPUT", "/corpus/derived")) / "tables"
WORK = Path(os.environ.get("SEMINAR_EXPLORATION_HOME", str(Path.home() / "explorations"))) / "covers"
MODEL = "facebook/dinov2-small"
MODEL_REVISION = "main"  # Shared snapshot, or resolve an exact commit for a private download.
SHARED_MODEL_ROOT = Path(os.environ.get("SEMINAR_MODELS", "/models")) / "exploration"
USE_SHARED_MODEL = SHARED_MODEL_ROOT.is_dir()  # False only for deliberate private model downloads.
DEVICE = os.environ.get("SEMINAR_DEVICE", "cpu")
BATCH_SIZE = 16
SEED = 42
MAX_POSTS = int(os.environ["SEMINAR_MAX_POSTS"]) if os.environ.get("SEMINAR_MAX_POSTS") else None
CLUSTER_METHOD = "hdbscan"  # Or kmeans, to compare a chosen number of groups.
MIN_CLUSTER_SIZE = 10
MIN_SAMPLES = 3
N_CLUSTERS = 8  # Used only by kmeans.
PCA_COMPONENTS = 50
MAP_NEIGHBOURS = 15
assert CLUSTER_METHOD in {"hdbscan", "kmeans"}
assert MAX_POSTS is None or MAX_POSTS >= 5
WORK.mkdir(parents=True, exist_ok=True)
CACHE = WORK / "cache"
CACHE.mkdir(exist_ok=True)
print("Results:", WORK)

In [ ]:
IDS = {"Post ID": str, "Text ID": str, "Image ID": str}
posts = pd.read_csv(TABLES / "posts.csv", dtype=IDS, keep_default_na=False)
images = pd.read_csv(TABLES / "images.csv", dtype=IDS, keep_default_na=False)
texts = pd.read_csv(TABLES / "texts.csv", dtype=IDS, keep_default_na=False)
assert posts["Post ID"].is_unique and images["Image ID"].is_unique and texts["Text ID"].is_unique
assert images["Post ID"].isin(posts["Post ID"]).all()
covers = images[images["Image Type"] == "Cover"].copy()
assert covers["Post ID"].is_unique, "Expected one cover per post."
coverage = posts.groupby("Username").size().rename("All posts").to_frame().join(
    covers.groupby("Username").size().rename("Covers")).fillna(0).astype(int)
coverage["Without cover"] = coverage["All posts"] - coverage["Covers"]
display(coverage)
print(f"{len(covers):,} covers from {len(posts):,} posts. Gallery images and extracted frames are excluded.")

captions = texts[texts["Text Type"] == "Caption"][["Post ID", "Text"]].rename(columns={"Text": "Caption"})
assert captions["Post ID"].is_unique
covers = covers.merge(posts[["Post ID", "Created", "Plays"]], on="Post ID", validate="one_to_one")
covers = covers.merge(captions, on="Post ID", how="left", validate="one_to_one").fillna({"Caption": ""})
covers = covers.sort_values("Image ID").reset_index(drop=True)
if MAX_POSTS is not None and len(covers) > MAX_POSTS:
    covers = covers.iloc[np.linspace(0, len(covers)-1, MAX_POSTS, dtype=int)].reset_index(drop=True)

# Report unreadable images and continue with the readable selection.
failures, valid = [], []
for _, row in covers.iterrows():
    try:
        with Image.open(row["Path"]) as image:
            image.verify()
        valid.append(True)
    except (OSError, ValueError) as error:
        valid.append(False)
        failures.append({"Image ID": row["Image ID"], "Post ID": row["Post ID"], "Error": str(error)})
covers = covers.loc[valid].reset_index(drop=True)
assert len(covers) >= 5, "Need at least five readable covers."
print(f"Selected {len(covers):,} readable covers; {len(failures)} failed.")
if failures:
    display(pd.DataFrame(failures))

## 2. Compute and cache visual embeddings

Each cover is padded to a square with a neutral background, preserving its aspect ratio and full contents. DINOv2 then resizes that square to its input resolution. This differs from centre-cropping a vertical TikTok cover: peripheral content remains visible, although fine text can become too small to read. Padding, text overlays and recurring layouts may all affect the grouping.

We use the whole-image **CLS token**, L2-normalized. Cache keys include image IDs, paths, file sizes/modification times, the exact model commit, package version and preprocessing method. Changed model inputs receive a new cache.

In [ ]:
import torch, transformers
from huggingface_hub import HfApi
from transformers import AutoImageProcessor, AutoModel
from tqdm.auto import tqdm

if USE_SHARED_MODEL:
    shared = (SHARED_MODEL_ROOT / MODEL.replace("/", "--") / "current").resolve(strict=True)
    manifest = json.loads((shared / "seminar-model.json").read_text())
    assert manifest["model"] == MODEL, "Shared model manifest does not match the selected model."
    revision = manifest["revision"]
    assert re.fullmatch(r"[0-9a-f]{40}", revision), "Expected an exact shared model revision."
    assert MODEL_REVISION in {"main", revision}, "Ask your instructor to prepare this revision, or use a private download."
    model_source = str(shared)
    load_options = {"local_files_only": True}
    print("Using read-only shared weights:", shared)
else:
    revision = MODEL_REVISION if re.fullmatch(r"[0-9a-f]{40}", MODEL_REVISION) else HfApi().model_info(MODEL, revision=MODEL_REVISION).sha
    model_source = MODEL
    load_options = {"revision": revision}
    print("No shared model selected: weights use your private Hugging Face cache.")
settings = {"model": MODEL, "revision": revision, "transformers": transformers.__version__,
            "preprocessing": "neutral_letterbox_448_then_processor_v1", "pooling": "CLS_L2"}
inputs_manifest = []
for _, row in covers.iterrows():
    stat = Path(row["Path"]).stat()
    inputs_manifest.append([row["Image ID"], row["Path"], stat.st_size, stat.st_mtime_ns])
fingerprint = hashlib.sha256(json.dumps({"settings": settings, "images": inputs_manifest}, sort_keys=True).encode()).hexdigest()
run_id = fingerprint[:16]
cache_path = CACHE / f"{fingerprint}.npz"
ids = covers["Image ID"].to_numpy(dtype=str)

if cache_path.exists():
    with np.load(cache_path, allow_pickle=False) as saved:
        assert np.array_equal(saved["ids"], ids)
        embeddings = saved["embeddings"]
    print("Loaded cached embeddings:", cache_path.name)
else:
    processor = AutoImageProcessor.from_pretrained(model_source, **load_options)
    encoder = AutoModel.from_pretrained(model_source, **load_options).to(DEVICE).eval()
    batches = []
    for start in tqdm(range(0, len(covers), BATCH_SIZE), desc="Embedding covers"):
        batch = []
        for path in covers["Path"].iloc[start:start+BATCH_SIZE]:
            with Image.open(path) as image:
                batch.append(ImageOps.pad(ImageOps.exif_transpose(image).convert("RGB"), (448, 448), color=(127, 127, 127)))
        inputs = processor(images=batch, return_tensors="pt").to(DEVICE)
        with torch.inference_mode():
            vectors = encoder(**inputs).last_hidden_state[:, 0, :]
            vectors = torch.nn.functional.normalize(vectors, p=2, dim=1)
        batches.append(vectors.cpu().numpy())
    embeddings = np.concatenate(batches).astype(np.float32)
    temporary = cache_path.with_suffix(".partial.npz")
    np.savez_compressed(temporary, ids=ids, embeddings=embeddings)
    temporary.replace(cache_path)
    del encoder, inputs, vectors
    import gc
    gc.collect()
    if DEVICE.startswith("cuda"):
        torch.cuda.empty_cache()
assert embeddings.ndim == 2 and len(embeddings) == len(covers) and np.isfinite(embeddings).all()
print("Embedding matrix:", embeddings.shape, "| exact model revision:", revision)

## 3. Cluster features, then make a browsing map

Clustering uses a PCA reduction of the normalized embeddings. The two-dimensional UMAP map is generated separately and never used to decide cluster membership.

- **HDBSCAN** can leave covers unassigned (`-1`). Change the minimum cluster size and inspect how stable the groups are.
- **k-means** assigns every cover to one of `N_CLUSTERS` groups. Compare choices; a chosen number of clusters is not a discovered number of frames.

Re-run from here after changing parameters. The cached image embeddings stay the same.

In [ ]:
from hdbscan import HDBSCAN
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from umap import UMAP

n = len(covers)
pca = PCA(n_components=min(PCA_COMPONENTS, n-1, embeddings.shape[1]), random_state=SEED)
features = pca.fit_transform(embeddings)
if CLUSTER_METHOD == "hdbscan":
    assert 2 <= MIN_CLUSTER_SIZE <= n
    clusters = HDBSCAN(min_cluster_size=MIN_CLUSTER_SIZE, min_samples=min(MIN_SAMPLES, n-1),
                       metric="euclidean").fit_predict(features)
else:
    assert 2 <= N_CLUSTERS < n
    clusters = KMeans(n_clusters=N_CLUSTERS, n_init=10, random_state=SEED).fit_predict(features)
coords = UMAP(n_components=2, n_neighbors=min(MAP_NEIGHBOURS, n-1), metric="cosine",
              random_state=SEED, n_jobs=1, init="random").fit_transform(embeddings)
results = covers.assign(Cluster=clusters, **{"Map X": coords[:, 0], "Map Y": coords[:, 1]})
print(f"PCA retains {pca.explained_variance_ratio_.sum():.1%} of variance.")
print(f"{len(set(clusters)-{-1})} clusters; {(clusters == -1).sum()} unassigned covers.")
display(results.groupby("Cluster").size().rename("Covers").to_frame())
display(pd.crosstab(results["Username"], results["Cluster"], normalize="index").round(2))

## 4. Browse the image map

Hover over a point for its cover and caption. Use **box selection**, **lasso selection** or a click to populate the gallery beneath the map. Switch colours between cluster and account: does a group mostly reflect one account's style? The viewer includes all selected covers as small thumbnails, and caps the displayed gallery at 60 items.

The initial gallery shows examples from the largest cluster, or from the unassigned group if everything is unassigned. The exported HTML works in a browser without Python or an internet connection.

In [ ]:
from bokeh.embed import file_html
from bokeh.layouts import column, row
from bokeh.models import ColumnDataSource, CustomJS, Div, HoverTool, Select
from bokeh.palettes import Turbo256
from bokeh.plotting import figure
from bokeh.resources import INLINE


def thumbnail(path):
    with Image.open(path) as image:
        image = ImageOps.exif_transpose(image).convert("RGB")
        image.thumbnail((150, 180))
        buffer = io.BytesIO()
        image.save(buffer, format="JPEG", quality=75)
    return "data:image/jpeg;base64," + base64.b64encode(buffer.getvalue()).decode("ascii")


def colours(values):
    labels = sorted(set(str(v) for v in values))
    palette = {label: Turbo256[round(j*255/max(1, len(labels)-1))] for j, label in enumerate(labels)}
    palette["-1"] = "#999999"
    return [palette[str(v)] for v in values]

cluster_colours = colours(results["Cluster"])
account_colours = colours(results["Username"])
source = ColumnDataSource(dict(
    x=coords[:, 0], y=coords[:, 1], colour=cluster_colours,
    cluster_colour=cluster_colours, account_colour=account_colours,
    cluster=results["Cluster"].astype(str).tolist(),
    account=results["Username"].map(html.escape).tolist(),
    post_id=results["Post ID"].tolist(), image_id=results["Image ID"].tolist(),
    caption=results["Caption"].map(lambda text: html.escape(text[:700])).tolist(),
    thumb=[thumbnail(path) for path in results["Path"]]))
p = figure(title="Cover similarity — nearby points invite comparison", height=500, sizing_mode="stretch_width",
           tools="pan,wheel_zoom,box_zoom,reset,box_select,lasso_select,tap,save", active_scroll="wheel_zoom")
p.scatter("x", "y", source=source, size=9, color="colour", alpha=0.85, selection_color="#111111", nonselection_alpha=0.18)
p.add_tools(HoverTool(tooltips="""<div style="max-width:300px"><img src="@thumb" style="max-height:180px"><br>
<b>@account{safe}</b> · Cluster @cluster<br>@post_id<br>@caption{safe}</div>"""))
p.axis.axis_label = "UMAP coordinate (no substantive unit)"
colour_choice = Select(title="Colour by", value="Cluster", options=["Cluster", "Account"], width=180)
colour_choice.js_on_change("value", CustomJS(args=dict(source=source), code="""
source.data.colour = source.data[cb_obj.value === "Account" ? "account_colour" : "cluster_colour"].slice();
source.change.emit();
"""))
largest = results["Cluster"].value_counts().index[0]
initial = np.flatnonzero(results["Cluster"].to_numpy() == largest)[:12].tolist()

def card(index):
    d = source.data
    return (f'<div style="width:190px;padding:8px;overflow-wrap:anywhere"><img src="{d["thumb"][index]}" style="max-height:180px;max-width:180px">'
            f'<br><b>{d["account"][index]}</b><br>Cluster {d["cluster"][index]}<br>{d["post_id"][index]}'
            f'<p style="font-size:12px">{d["caption"][index]}</p></div>')

gallery = Div(text='<div style="display:flex;flex-wrap:wrap">'+''.join(card(j) for j in initial)+'</div>', sizing_mode="stretch_width")
info = Div(text=f"Initial examples: cluster {largest}. Select points to replace this gallery.", sizing_mode="stretch_width")
source.selected.js_on_change("indices", CustomJS(args=dict(source=source, gallery=gallery, info=info), code="""
const indices = source.selected.indices;
const d = source.data;
info.text = `${indices.length} selected; showing up to 60 covers.`;
gallery.text = '<div style="display:flex;flex-wrap:wrap">' + indices.slice(0,60).map(i =>
`<div style="width:190px;padding:8px;overflow-wrap:anywhere"><img src="${d.thumb[i]}" style="max-height:180px;max-width:180px">
<br><b>${d.account[i]}</b><br>Cluster ${d.cluster[i]}<br>${d.post_id[i]}<p style="font-size:12px">${d.caption[i]}</p></div>`
).join('') + '</div>';
"""))
source.name = "cover_source"
gallery.name = "cover_gallery"
info.name = "selection_info"
colour_choice.name = "colour_by"
viewer_html = file_html(column(row(colour_choice), p, info, gallery, sizing_mode="stretch_width"), INLINE, "Explore TikTok covers")
# Inline iframe avoids any extra server/port and uses the same viewer as the export.
display(HTML('<iframe srcdoc="'+html.escape(viewer_html, quote=True)+'" style="width:100%;height:950px;border:1px solid #ddd"></iframe>'))

## 5. Inspect nearest neighbours

The map can distort distances. For one cover, inspect its nearest neighbours using cosine similarity in the **original embedding space**. Scores describe this model's similarity, not shared framing or truthfulness.

In [ ]:
QUERY_IMAGE_ID = results.iloc[0]["Image ID"]  # Replace with an Image ID from the map.
query = results.index[results["Image ID"] == QUERY_IMAGE_ID]
assert len(query) == 1, "Choose an Image ID in this selection."
similarity = embeddings @ embeddings[query[0]]
nearest = np.argsort(-similarity)
nearest = nearest[nearest != query[0]][:8]
display(results.iloc[nearest][["Image ID", "Post ID", "Username", "Cluster"]].assign(Similarity=similarity[nearest]))
display(HTML('<p>Query</p>'+card(int(query[0]))+'<p>Nearest neighbours</p><div style="display:flex;flex-wrap:wrap">'+''.join(card(int(j)) for j in nearest)+'</div>'))

## 6. Save the map and assignments

The embedding cache is separate from the run folder. Different clustering/map parameters create a new run folder; captions and post metadata are included in its fingerprint so refreshed viewer content also gets a new folder. All exports contain corpus material and belong outside the public code repository.

In [ ]:
run_settings = dict(settings, fingerprint=fingerprint, clustering=CLUSTER_METHOD,
                    min_cluster_size=MIN_CLUSTER_SIZE, min_samples=MIN_SAMPLES, n_clusters=N_CLUSTERS,
                    pca_components=PCA_COMPONENTS, map_neighbours=MAP_NEIGHBOURS, seed=SEED)
parameter_id = hashlib.sha256(json.dumps({"settings": run_settings,
    "metadata": results[["Image ID", "Username", "Caption", "Created", "Plays"]].astype(str).values.tolist()}, sort_keys=True).encode()).hexdigest()[:10]
RUN = WORK / f"{run_id}-{parameter_id}"
RUN.mkdir(exist_ok=True)
(RUN / "cover-map.html").write_text(viewer_html, encoding="utf-8")
results.to_csv(RUN / "image_clusters.csv", index=False)
coverage.to_csv(RUN / "cover_coverage.csv")
pd.DataFrame(failures, columns=["Image ID", "Post ID", "Error"]).to_csv(RUN / "unreadable_images.csv", index=False)
(RUN / "settings.json").write_text(json.dumps(run_settings, indent=2), encoding="utf-8")
print("Saved:", RUN)
# JupyterHub serves files relative to the home folder, not their absolute disk path.
from urllib.parse import quote
export = RUN / "cover-map.html"
prefix = os.environ.get("JUPYTERHUB_SERVICE_PREFIX")
if prefix and export.is_relative_to(Path.home()):
    url = prefix.rstrip("/") + "/files/" + quote(str(export.relative_to(Path.home())))
    display(HTML(f'<a href="{html.escape(url, quote=True)}" target="_blank">Open exported map</a>'))
else:
    display(FileLink(os.path.relpath(export, Path.cwd())))

## Questions for your first exploration

- Select five nearby covers. Which visible features connect them? Find a counterexample.
- Compare colouring by cluster and account. Does an apparent visual pattern mainly belong to one account?
- Does the model group subjects, composition, colour, image style, text overlays, or something else?
- Which distinctions matter for your research question but are absent from this map?
- Compare covers with linked text. What context is missing from a single cover, and what would you need to inspect in the video?
- Record tentative observations for a codebook. Do not use the clusters as validated frame labels.

**Before committing this notebook:** clear all outputs. The map embeds corpus thumbnails and captions, even though the HTML is self-contained.